In [ ]:
# transformers may be old on Kaggle, so update it. torch/torchvision already exist.
!pip install -q -U transformers

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval()
print("CLIP loaded.")

In [ ]:
# Load CIFAR-10 test set (as PIL images, no transform)

test_dataset = datasets.CIFAR10(root="./data", train=False, download=True)

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Total test images:", len(test_dataset))

In [ ]:
# Zero-shot classification with CLIP

text_prompts = [f"a photo of a {c}" for c in class_names]


NUM_IMAGES = len(test_dataset)   
BATCH_SIZE = 256

all_preds = []
all_labels = []

for start in range(0, NUM_IMAGES, BATCH_SIZE):
    end = min(start + BATCH_SIZE, NUM_IMAGES)
    batch_imgs = [test_dataset[i][0] for i in range(start, end)]      # PIL images
    batch_labels = [test_dataset[i][1] for i in range(start, end)]    # true labels

    inputs = processor(text=text_prompts, images=batch_imgs,
                       return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits_per_image = outputs.logits_per_image      # shape: [batch, 10]
        preds = logits_per_image.argmax(dim=1).cpu().numpy()

    all_preds.extend(preds.tolist())
    all_labels.extend(batch_labels)
    print(f"Processed {end}/{NUM_IMAGES}")

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
print("Done.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(all_labels, all_preds)
print(f"CLIP Zero-shot Accuracy: {acc*100:.2f}%\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(9, 9))
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("CLIP Zero-shot — Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:

prompt_templates = [
    "{}",
    "a photo of a {}",
    "a blurry photo of a {}",
    "a photo of the small {}",
    "a picture of a {}",
]

SUBSET = 2000  
sub_labels = [test_dataset[i][1] for i in range(SUBSET)]
sub_imgs   = [test_dataset[i][0] for i in range(SUBSET)]

for template in prompt_templates:
    prompts = [template.format(c) for c in class_names]
    preds = []
    for start in range(0, SUBSET, BATCH_SIZE):
        end = min(start + BATCH_SIZE, SUBSET)
        inputs = processor(text=prompts, images=sub_imgs[start:end],
                           return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            out = model(**inputs)
            preds.extend(out.logits_per_image.argmax(dim=1).cpu().numpy().tolist())
    a = accuracy_score(sub_labels, preds)
    print(f"Template '{template}'  ->  Accuracy: {a*100:.2f}%")

In [ ]:
#Error analysis 

wrong_idx = np.where(all_labels != all_preds)[0]
print(f"Total wrong: {len(wrong_idx)} out of {len(all_labels)}")

# Show 8 misclassified images with true vs predicted labels
plt.figure(figsize=(14, 7))
for plot_i, idx in enumerate(wrong_idx[:8]):
    img = test_dataset[idx][0]
    plt.subplot(2, 4, plot_i + 1)
    plt.imshow(img)
    plt.title(f"True: {class_names[all_labels[idx]]}\nPred: {class_names[all_preds[idx]]}",
              fontsize=10)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
#Load CIFAR-10 with transforms and train ResNet18

import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader

# ResNet expects normalized 224x224 images
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.CIFAR10(root="./data", train=True,  download=True, transform=transform)
test_ds  = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2)

cnn = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
cnn.fc = nn.Linear(cnn.fc.in_features, 10)   
cnn = cnn.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=1e-4)

EPOCHS = 3   # 3 epochs is enough to beat/compare with zero-shot
for epoch in range(EPOCHS):
    cnn.train()
    running = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(cnn(imgs), labels)
        loss.backward()
        optimizer.step()
        running += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}  avg loss: {running/len(train_loader):.4f}")
print("CNN training done.")

In [ ]:
#Evaluate the CNN

cnn.eval()
cnn_preds, cnn_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out = cnn(imgs)
        cnn_preds.extend(out.argmax(dim=1).cpu().numpy().tolist())
        cnn_labels.extend(labels.numpy().tolist())

cnn_acc = accuracy_score(cnn_labels, cnn_preds)
print(f"Fine-tuned CNN Accuracy: {cnn_acc*100:.2f}%")

In [ ]:
#Final comparison table

print("="*45)
print(f"{'Model':<22}{'Accuracy':<12}{'Training?'}")
print("="*45)
print(f"{'CLIP Zero-shot':<22}{acc*100:>6.2f}%     No")
print(f"{'Fine-tuned ResNet18':<22}{cnn_acc*100:>6.2f}%     Yes")
print("="*45)

In [ ]:
import random

# Pick a random image 
idx = random.randint(0, len(test_dataset) - 1)
img, true_label = test_dataset[idx]

plt.imshow(img)
plt.axis("off")
plt.title(f"True label: {class_names[true_label]}")
plt.show()

inputs = processor(text=text_prompts, images=img,
                   return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)[0]

pred_idx = probs.argmax().item()
result = "Correct" if pred_idx == true_label else " Wrong"
print(f"Prediction: {class_names[pred_idx]}  ({probs[pred_idx]*100:.2f}%)  {result}\n")

print("All class probabilities:")
for i in probs.argsort(descending=True):
    print(f"  {class_names[i]:<12} {probs[i]*100:5.2f}%")

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count("/") - "/kaggle/input".count("/")
    if depth <= 4:
        print(root)

In [ ]:
#The reusable pipeline function 

def run_clip_zeroshot(data_dir, prompt_template="a photo of a {}",
                      batch_size=256, n_error_examples=8, title=""):
   
    # Load images 
    dataset = datasets.ImageFolder(root=data_dir)
    class_names = dataset.classes
    print(f"[{title}] Classes: {class_names} | Total images: {len(dataset)}")

    text_prompts = [prompt_template.format(c) for c in class_names]

    # Zero-shot prediction
    all_preds, all_labels = [], []
    for start in range(0, len(dataset), batch_size):
        end = min(start + batch_size, len(dataset))
        imgs   = [dataset[i][0].convert("RGB") for i in range(start, end)]
        labels = [dataset[i][1] for i in range(start, end)]
        inputs = processor(text=text_prompts, images=imgs,
                           return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            preds = model(**inputs).logits_per_image.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels)
        print(f"  Processed {end}/{len(dataset)}")

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    #  Accuracy + report 
    acc = accuracy_score(all_labels, all_preds)
    print(f"\n[{title}] CLIP Zero-shot Accuracy: {acc*100:.2f}%\n")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    #  Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
    plt.title(f"{title} — CLIP Zero-shot Confusion Matrix")
    plt.tight_layout()
    plt.show()

    #  Error analysis: show a few wrong predictions 
    wrong_idx = np.where(all_labels != all_preds)[0]
    print(f"[{title}] Total wrong: {len(wrong_idx)} / {len(all_labels)}")
    if len(wrong_idx) > 0:
        plt.figure(figsize=(14, 7))
        for plot_i, idx in enumerate(wrong_idx[:n_error_examples]):
            img = dataset[idx][0].convert("RGB")
            plt.subplot(2, 4, plot_i + 1)
            plt.imshow(img)
            plt.title(f"True: {class_names[all_labels[idx]]}\n"
                      f"Pred: {class_names[all_preds[idx]]}", fontsize=10)
            plt.axis("off")
        plt.tight_layout()
        plt.show()

    return acc, all_preds, all_labels, class_names

In [ ]:
#Helper function to auto-find the right folder

import os

def find_image_folder(start_path, min_classes=2):
    
    current = start_path
    while True:
        items = [i for i in os.listdir(current) if not i.startswith('.')]
        subdirs = [i for i in items if os.path.isdir(os.path.join(current, i))]

        if len(subdirs) >= min_classes:
            # Check subdirs actually contain image files (real class folders)
            sample_dir = os.path.join(current, subdirs[0])
            sample_files = os.listdir(sample_dir)
            has_images = any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in sample_files)
            if has_images:
                return current

        if len(subdirs) == 1:
            # Only one folder inside -> go one level deeper
            current = os.path.join(current, subdirs[0])
        else:
            # Can't resolve automatically
            raise Exception(f"Could not auto-detect class folder. Stopped at: {current}, found: {items}")

INTEL_DIR = find_image_folder("/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test")
FLOWERS_DIR = find_image_folder("/kaggle/input/datasets/alxmamaev/flowers-recognition/flowers")

print("Intel dir:", INTEL_DIR)
print("Flowers dir:", FLOWERS_DIR)

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

In [ ]:
#Run on Intel dataset

intel_acc, intel_preds, intel_labels, intel_classes = run_clip_zeroshot(
    data_dir=INTEL_DIR,
    prompt_template="a photo of a {}",
    title="Intel"
)

**Intel CNN**

In [ ]:
# Intel has ready-made train/test folders, 
INTEL_TRAIN_DIR = find_image_folder("/kaggle/input/datasets/puneet6060/intel-image-classification/seg_train")

transform = transforms.Compose([
    transforms.Resize((224, 224)),   # forces exact square size, ignoring aspect ratio
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

intel_train_ds = datasets.ImageFolder(root=INTEL_TRAIN_DIR, transform=transform)
intel_test_ds  = datasets.ImageFolder(root=INTEL_DIR, transform=transform)

intel_train_loader = DataLoader(intel_train_ds, batch_size=128, shuffle=True, num_workers=2)
intel_test_loader  = DataLoader(intel_test_ds, batch_size=256, shuffle=False, num_workers=2)

intel_cnn = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
intel_cnn.fc = nn.Linear(intel_cnn.fc.in_features, len(intel_train_ds.classes))
intel_cnn = intel_cnn.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(intel_cnn.parameters(), lr=1e-4)

EPOCHS = 3
for epoch in range(EPOCHS):
    intel_cnn.train()
    running = 0.0
    for imgs, labels in intel_train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(intel_cnn(imgs), labels)
        loss.backward()
        optimizer.step()
        running += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}  avg loss: {running/len(intel_train_loader):.4f}")

intel_cnn.eval()
intel_cnn_preds, intel_cnn_labels = [], []
with torch.no_grad():
    for imgs, labels in intel_test_loader:
        imgs = imgs.to(device)
        out = intel_cnn(imgs)
        intel_cnn_preds.extend(out.argmax(dim=1).cpu().numpy().tolist())
        intel_cnn_labels.extend(labels.numpy().tolist())

intel_cnn_acc = accuracy_score(intel_cnn_labels, intel_cnn_preds)
print(f"Intel Fine-tuned CNN Accuracy: {intel_cnn_acc*100:.2f}%")

In [ ]:
#Run on Flowers dataset

flowers_acc, flowers_preds, flowers_labels, flowers_classes = run_clip_zeroshot(
    data_dir=FLOWERS_DIR,
    prompt_template="a photo of a {}, a type of flower",
    title="Flowers"
)

**Flowers CNN**

In [ ]:
from torch.utils.data import random_split

flowers_full_ds = datasets.ImageFolder(root=FLOWERS_DIR, transform=transform)

torch.manual_seed(42)  # for reproducibility
train_size = int(0.8 * len(flowers_full_ds))
test_size  = len(flowers_full_ds) - train_size
flowers_train_ds, flowers_test_ds = random_split(flowers_full_ds, [train_size, test_size])

flowers_train_loader = DataLoader(flowers_train_ds, batch_size=128, shuffle=True, num_workers=2)
flowers_test_loader  = DataLoader(flowers_test_ds, batch_size=256, shuffle=False, num_workers=2)

flowers_cnn = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
flowers_cnn.fc = nn.Linear(flowers_cnn.fc.in_features, len(flowers_full_ds.classes))
flowers_cnn = flowers_cnn.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(flowers_cnn.parameters(), lr=1e-4)

EPOCHS = 3
for epoch in range(EPOCHS):
    flowers_cnn.train()
    running = 0.0
    for imgs, labels in flowers_train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(flowers_cnn(imgs), labels)
        loss.backward()
        optimizer.step()
        running += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}  avg loss: {running/len(flowers_train_loader):.4f}")

flowers_cnn.eval()
flowers_cnn_preds, flowers_cnn_labels = [], []
with torch.no_grad():
    for imgs, labels in flowers_test_loader:
        imgs = imgs.to(device)
        out = flowers_cnn(imgs)
        flowers_cnn_preds.extend(out.argmax(dim=1).cpu().numpy().tolist())
        flowers_cnn_labels.extend(labels.numpy().tolist())

flowers_cnn_acc = accuracy_score(flowers_cnn_labels, flowers_cnn_preds)
print(f"Flowers Fine-tuned CNN Accuracy: {flowers_cnn_acc*100:.2f}%")

In [ ]:
#final comparison table

cifar_acc       = 0.8880
cnn_acc         = 0.9415
intel_acc       = 0.9300
intel_cnn_acc   = 0.9247
flowers_acc     = 0.9479
flowers_cnn_acc = 0.9248

intel_classes   = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
flowers_classes = ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']

print("="*65)
print(f"{'Dataset':<12}{'Classes':<10}{'Zero-shot':<14}{'Fine-tuned CNN'}")
print("="*65)
print(f"{'CIFAR-10':<12}{'10':<10}{cifar_acc*100:>7.2f}%      {cnn_acc*100:>7.2f}%")
print(f"{'Intel':<12}{len(intel_classes):<10}{intel_acc*100:>7.2f}%      {intel_cnn_acc*100:>7.2f}%")
print(f"{'Flowers':<12}{len(flowers_classes):<10}{flowers_acc*100:>7.2f}%      {flowers_cnn_acc*100:>7.2f}%")
print("="*65)

In [ ]:
# Quick comparison: plain prompt vs flower-aware prompt, on a subset for speed
flowers_dataset = datasets.ImageFolder(root=FLOWERS_DIR)
SUBSET = 500
sub_imgs = [flowers_dataset[i][0].convert("RGB") for i in range(SUBSET)]
sub_labels = [flowers_dataset[i][1] for i in range(SUBSET)]

for template in ["a photo of a {}", "a photo of a {}, a type of flower"]:
    prompts = [template.format(c) for c in flowers_dataset.classes]
    inputs = processor(text=prompts, images=sub_imgs, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        preds = model(**inputs).logits_per_image.argmax(dim=1).cpu().numpy()
    a = accuracy_score(sub_labels, preds)
    print(f"'{template}' -> {a*100:.2f}%")

In [ ]:
#Save these to results file
with open("/kaggle/working/final_results.txt", "w") as f:
    f.write("Dataset       Zero-shot    Fine-tuned CNN\n")
    f.write(f"CIFAR-10      {cifar_acc*100:.2f}%       {cnn_acc*100:.2f}%\n")
    f.write(f"Intel         {intel_acc*100:.2f}%       {intel_cnn_acc*100:.2f}%\n")
    f.write(f"Flowers       {flowers_acc*100:.2f}%       {flowers_cnn_acc*100:.2f}%\n")
print("Saved.")